In [ ]:
#| default_exp cli.poll

In [ ]:
#| export
from fastcore.script import *
from fastcore.style import S
from rich.console import Console
from rich.table import Table
from marisco.configs import ZENODO_COMMUNITY
from marisco.poll import handler_records, check_records, new_datasets

In [ ]:
#| export
def _report(df, none_msg, some_msg):
    "Print `none_msg` if `df` is empty, else `some_msg` and `df`"
    if df.empty: return print(S.green(none_msg))
    print(S.yellow(some_msg))
    tbl = Table(*df.columns)
    for row in df.astype(str).itertuples(index=False): tbl.add_row(*row)
    Console().print(tbl)

In [ ]:
#| export
@call_parse
def main(
    community: str=ZENODO_COMMUNITY, # Zenodo community to look for new datasets in
) -> None:
    "Check the Zenodo records that handlers read for newer versions, and list the datasets in a Zenodo community that no handler reads"
    print('Checking the Zenodo records that handlers read ...')
    checked = check_records(handler_records())
    old = checked.loc[~checked.current, ['handler', 'key', 'record_id', 'latest_id']]
    _report(old, 'All records are current.', f'{len(old)} record(s) have a newer version on Zenodo:')
    new = new_datasets(checked, community)[['record_id', 'created', 'title']]
    _report(new, f'No new datasets in community {community}.', f'{len(new)} new dataset(s) in community {community}:')

## Usage

`marisco-poll` checks the Zenodo records that handlers read. It lists each record whose dataset has a newer version, with the id of that version, and each dataset in the Zenodo community that no handler reads. `--community` selects another community than `titanica`.

    marisco-poll
    marisco-poll --community titanica